In [1]:
!pip install scikit-learn

In [2]:
import re
import datetime
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

In [3]:
CONV_SAMPLES = {
    "greetings": [
        "Hi", "hello", "How are you", "hey there", "hey", "Hi"
    ],
    "taxi": [
        "book a cab", "need a ride", "find me a cab"
    ],
    "weather": [
        "what is the weather in tokyo",
        "weather germany",
        "what is the weather like in kochi",
        "what is the weather like",
        "is it hot outside"
    ],
    "datetime": [
        "what day is today",
        "todays date",
        "what time is it now",
        "time now",
        "what is the time"
    ],
    "music": [
        "play the Beatles",
        "shuffle songs",
        "make a sound"
    ]
}

X = []
y = []

for intent, samples in CONV_SAMPLES.items():
    for sample in samples:
        X.append(sample)
        y.append(intent)

clf = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("classifier", MultinomialNB())
])

clf.fit(X, y)

print("Intent classifier trained")

Intent classifier trained


In [4]:
print(clf.predict(["will it rain today"])[0])
print(clf.predict(["play playlist rock n roll"])[0])
print(clf.predict(["what's the hour?"])[0])
print(clf.predict(["hello there"])[0])

datetime
music
weather
greetings


In [5]:
class EntityExtractor:
    def fit(self, X, y):
        self.samples = X
        self.entities = y
        return self

    def predict(self, text):
        text_lower = text.lower()
        
        for sample, entity in zip(self.samples, self.entities):
            sample_words = sample.lower().split()
            
            for key, value in entity.items():
                if key == "place" and value.lower() in text_lower:
                    return entity
                if key == "greet" and value.lower() in text_lower:
                    return entity
                if key == "target" and value.lower() in text_lower:
                    return entity
        
        return {}

In [6]:
X_WEATHER = [
    "what is the weather in tokyo",
    "weather germany",
    "what is the weather like in kochi",
    "what is the weather in London"
]

Y_WEATHER = [
    {"intent": "weather", "place": "tokyo"},
    {"intent": "weather", "place": "germany"},
    {"intent": "weather", "place": "kochi"},
    {"intent": "weather", "place": "London"}
]

EX_WEATHER = EntityExtractor()
EX_WEATHER.fit(X_WEATHER, Y_WEATHER)

print(EX_WEATHER.predict("what is the weather in London"))

{'intent': 'weather', 'place': 'London'}


In [7]:
X_GREETING = ["hi", "hello", "howdy", "hey there", "hey", "Hi"]
Y_GREETING = [
    {"greet": "Hi"},
    {"greet": "hello"},
    {"greet": "howdy"},
    {"greet": "hey"},
    {"greet": "hey"},
    {"greet": "Hi"}
]

EX_GREETING = EntityExtractor()
EX_GREETING.fit(X_GREETING, Y_GREETING)

X_DATETIME = [
    "what day is today",
    "date today",
    "what time is it now",
    "time now"
]

Y_DATETIME = [
    {"intent": "day", "target": "today"},
    {"intent": "date", "target": "today"},
    {"intent": "time", "target": "now"},
    {"intent": "time", "target": "now"}
]

EX_DATETIME = EntityExtractor()
EX_DATETIME.fit(X_DATETIME, Y_DATETIME)

print(EX_GREETING.predict("hello there"))
print(EX_DATETIME.predict("what time is it now"))

{'greet': 'hello'}
{'intent': 'time', 'target': 'now'}


In [8]:
def get_weather_forecast(place):
    sample_weather = {
        "london": "overcast clouds",
        "tokyo": "clear sky",
        "germany": "light rain",
        "kochi": "humid weather"
    }

    place_clean = place.lower()
    return sample_weather.get(place_clean, "weather information is not available")

In [9]:
_EXTRACTORS = {
    "taxi": None,
    "weather": EX_WEATHER,
    "greetings": EX_GREETING,
    "datetime": EX_DATETIME,
    "music": None
}

def question_and_answer(u_query: str):
    q_class = clf.predict([u_query])[0]
    print("Predicted intent:", q_class)

    if _EXTRACTORS[q_class] is None:
        return "Sorry, you have to upgrade your software!"

    q_entities = _EXTRACTORS[q_class].predict(u_query)
    print("Extracted entities:", q_entities)

    if q_class == "greetings":
        return q_entities.get("greet", "hello")

    if q_class == "weather":
        place = q_entities.get("place", "London").replace("_", " ")
        return "The forecast for {} is {}".format(
            place,
            get_weather_forecast(place)
        )

    if q_class == "datetime":
        return "Today's date is {}".format(
            datetime.datetime.today().strftime("%B %d, %Y")
        )

    return "I could not understand what you said. I am sorry."

In [10]:
print(question_and_answer("hello"))
print()
print(question_and_answer("what is the weather in London"))
print()
print(question_and_answer("what time is it now"))
print()
print(question_and_answer("play the Beatles"))

Predicted intent: greetings
Extracted entities: {'greet': 'hello'}
hello

Predicted intent: weather
Extracted entities: {'intent': 'weather', 'place': 'London'}
The forecast for London is overcast clouds

Predicted intent: datetime
Extracted entities: {'intent': 'time', 'target': 'now'}
Today's date is June 11, 2026

Predicted intent: music
Sorry, you have to upgrade your software!
